# ProcessBehavior Tutorial: Fill Weight Analysis

This notebook demonstrates **iterative process behavior analysis** using fill weight data from a 4-lane filling system.

## What We'll Cover

### Part 1: Start Simple - Lane Analysis
- Load and explore data
- Analyze by lane only
- Understand Xbar/S charts and appropriate WECO rules
- Detect signals correctly

### Part 2: Go Deeper - Lane + Phase Analysis  
- Add fill cycle phase (up/down arm position)
- Full variance decomposition with residuals
- Main effects and interactions
- Export comprehensive results

## The Data

**Filling System**: 4 lanes, each with up/down fill arm motion
- **pull**: Production sequence (1-100 time points)
- **lane**: Filling lane (1-4)
- **phase**: Fill cycle position (1=down, 2=up)
- **fill_weight**: Target measurement

---

# Part 1: Start Simple - Analyze by Lane

## Step 1: Load and Explore Data

In [1]:
# Import required libraries
import pandas as pd
from processbehavior import ProcessBehavior
from processbehavior.signals import SignalConfig
from pathlib import Path

# Find the data file (works from any directory)
data_paths = [
    'processbehavior/datasets/data/FILLWEIGHTDATA_800.csv',  # Running from project root
    '../../processbehavior/datasets/data/FILLWEIGHTDATA_800.csv',  # Running from examples/tom/
]

df = None
for path in data_paths:
    if Path(path).exists():
        df = pd.read_csv(path)
        print(f"Loaded data from: {path}")
        break

if df is None:
    raise FileNotFoundError(
        "Could not find FILLWEIGHTDATA_800.csv. "
        "Make sure you're running this notebook from the project root or examples/tom/ directory."
    )

# Explore the data
print(f"\nDataset: {len(df)} observations")
print(f"  Pulls (time points): {df['pull'].nunique()}")
print(f"  Lanes: {sorted(df['lane'].unique())}")
print(f"  Phases: {sorted(df['phase'].unique())}")
print(f"  Missing values: {df['fill_weight'].isna().sum()}")

df.head(12)  # Show first 12 rows (3 pulls × 4 lanes)

Loaded data from: ../../processbehavior/datasets/data/FILLWEIGHTDATA_800.csv

Dataset: 800 observations
  Pulls (time points): 100
  Lanes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Phases: [np.int64(1), np.int64(2)]
  Missing values: 11


,pull,lane,phase,fill_weight
0,1,1,1,236.93
1,1,1,2,237.39
2,1,2,1,236.30
3,1,2,2,241.35
4,1,3,1,236.09
5,1,3,2,232.30
6,1,4,1,235.81
7,1,4,2,241.89
8,2,1,1,239.67
9,2,1,2,236.39


## Step 2: Create ProcessDataFrame

The `ProcessDataFrame` automatically handles missing values and garbage characters in your data.

In [2]:
# Create ProcessBehavior - automatically cleans data
pb = ProcessBehavior(df)

print(f"ProcessBehavior ready with {len(pb.data)} observations")
print(f"  Cleaned/removed: {len(df) - len(pb.data)} rows with missing values")

ProcessBehavior ready with 800 observations
  Cleaned/removed: 0 rows with missing values


## Step 3: Analyze by Lane Only

**Question**: Are the 4 lanes performing consistently over time?

We'll group by `lane` only, treating phase as replicates within each lane/pull combination.

In [3]:
# Run lane-only analysis using formulate() -> execute()
study_lane = pb.formulate(
    response=pb.cols.fill_weight,
    factors=[pb.cols.lane],  # Group by lane only
    time=pb.cols.pull        # Time sequence
)

result_lane = study_lane.execute()

print("\n" + "=" * 80)
print("LANE ANALYSIS SUMMARY")
print("=" * 80)
print(f"SDS Type: {study_lane.sds}")
print(f"Recommended Chart: {study_lane.recommended_chart}")
print(f"Valid Charts: {study_lane.valid_charts}")
print(f"Charts Created: {result_lane.all_charts}")
print(f"Observations: {result_lane.summary['n_observations']}")


LANE ANALYSIS SUMMARY
SDS Type: 1
Recommended Chart: Xbar
Valid Charts: ['Xbar', 'S', 'R', 'Imr']
Charts Created: ['Xbar', 'S']
Observations: 789


### Understanding the Results

**SDS 1**: Full Replication
- 4 lanes (subgroups) with ~200 observations per lane across 100 time points
- Each lane has n≈200 observations (100 pulls × ~2 phases)
- True within-subgroup variance can be estimated
- Most statistically powerful design

**Available Charts**:
- **Xbar**: Average fill weight by lane over time
- **Sbar**: Variation within each lane over time

**Important**: These charts show **categorical comparisons** (lane 1 vs lane 2 vs lane 3 vs lane 4), not individual sequential measurements over time.

## Step 4: Visualize Lane Performance

In [4]:
# Plot Xbar and S charts
fig_lane = result_lane.plot(
    template='processbehavior',
    width=1400,
    height=700,
    highlight_signals=True
)

fig_lane.show()

**Interpretation Tips**:
- **Xbar Chart**: Shows if lane averages are stable over time
- **Sbar Chart**: Shows if lane variation is consistent
- Points beyond limits indicate special causes
- **Interactive**: Hover over points to see details!

## Step 5: Understanding WECO Rules for Xbar/S Charts

### Important Distinction: Chart Types and Rule Applicability

**Xbar and S charts** show categorical comparisons of rational subgroups:
- Each point = mean (or stddev) of a subgroup
- Points represent different categories (lanes), not sequential individual measurements
- Consecutive points on the chart are NOT in temporal sequence

**WECO Rules Applicability**:
- ✅ **Rule 1** (Point beyond control limits): Valid for all chart types
- ❌ **Rules 2-4** (Sequential patterns): Only valid for time series charts (IMR, R)
  - Rule 2: "2 of 3 consecutive" - needs temporal sequence
  - Rule 3: "4 of 5 consecutive" - needs temporal sequence  
  - Rule 4: "8+ run" - needs temporal sequence

**Why this matters**: Applying sequential rules to categorical comparisons produces meaningless results. The lanes are not "consecutive" in any temporal sense.

**For time-series analysis**: Use stratified IMR charts (shown in Part 2).

## Step 6: Detect Signals (Correct Usage)

In [5]:
# Detect signals using only Rule 1 (appropriate for Xbar/S charts)
signals_xbar_lane = result_lane.detect_signals(
    chart='Xbar',
    rules='default',  # Uses chart-type-based defaults (Rule 1 only for Xbar)
    config=SignalConfig(min_observations=2)  # Lowered for tutorial
)

signals_sbar_lane = result_lane.detect_signals(
    chart='S',
    rules='default',  # Uses chart-type-based defaults (Rule 1 only for S)
    config=SignalConfig(min_observations=2)
)

print("Signal Detection - Lane Analysis:")
print(f"  Xbar (means): {signals_xbar_lane.count} signals (Rule 1 only)")
print(f"  Sbar (variation): {signals_sbar_lane.count} signals (Rule 1 only)")
print("\nNote: Only Rule 1 applied - appropriate for categorical comparisons")

if signals_xbar_lane.has_signals:
    print(f"\nXbar Signals:")
    print(signals_xbar_lane.violations[['rule_name', 'value', 'description']].head(10))

Signal Detection - Lane Analysis:
  Xbar (means): 3 signals (Rule 1 only)
  Sbar (variation): 2 signals (Rule 1 only)

Note: Only Rule 1 applied - appropriate for categorical comparisons

Xbar Signals:
  rule_name    value                  description
0    rule_1  238.398  Point beyond control limits
1    rule_1  238.162  Point beyond control limits
2    rule_1  236.727  Point beyond control limits


## Step 7: Check Lane Main Effects

Are some lanes systematically different from others?

In [6]:
if result_lane.has_effects and 'lane' in result_lane.effects:
    print("Lane Main Effects:")
    print(result_lane.effects['lane'])
    print("\nInterpretation:")
    print("  • 'Main_Effect' shows deviation from grand mean")
    print("  • Positive = higher than average")
    print("  • Negative = lower than average")
else:
    print("Main effects not available for this configuration")

Lane Main Effects:
   lane  Main_Effect
0     1     0.615803
1     2     0.379297
2     3    -1.055162
3     4     0.058138

Interpretation:
  • 'Main_Effect' shows deviation from grand mean
  • Positive = higher than average
  • Negative = lower than average


---

# Part 2: Go Deeper - Add Phase Analysis

## Step 8: Analyze Lane + Phase

**Question**: Does the fill arm position (up/down) affect fill weight differently across lanes?

Now we'll include **both lane and phase** as grouping variables to see the full picture.

In [7]:
# Run lane + phase analysis using formulate() -> execute()
study_full = pb.formulate(
    response=pb.cols.fill_weight,
    factors=[pb.cols.lane, pb.cols.phase],  # Both factors
    time=pb.cols.pull
)

result_full = study_full.execute()

print("\n" + "=" * 80)
print("LANE + PHASE ANALYSIS SUMMARY")
print("=" * 80)
print(f"SDS Type: {study_full.sds}")
print(f"Recommended Chart: {study_full.recommended_chart}")
print(f"Valid Charts: {study_full.valid_charts}")
print(f"Charts Created: {result_full.all_charts}")
print(f"Observations: {result_full.summary['n_observations']}")
print(f"\nCapabilities:")
print(f"  Residuals: {result_full.has_residuals}")
print(f"  Main Effects: {result_full.has_effects}")
print(f"  Interactions: {result_full.has_interactions}")


LANE + PHASE ANALYSIS SUMMARY
SDS Type: 1
Recommended Chart: Xbar
Valid Charts: ['Xbar', 'S', 'R', 'Imr']
Charts Created: ['Xbar', 'S']
Observations: 789

Capabilities:
  Residuals: True
  Main Effects: True
  Interactions: True


### Understanding the Change

**SDS 1**: Full Replication
- 8 subgroups (4 lanes × 2 phases) with ~100 observations each across 100 time points  
- Each (lane × phase) combination has n≈100 observations
- True within-subgroup variance can be estimated
- Full main effects and interaction analysis available

**Note on Cell Definition**:
- Subgroups = (lane × phase) combinations (8 total)
- Sample size n = observations per subgroup **across all time points**
- time_var is for sequencing/plotting, not subgroup definition
- This allows proper SDS classification and S chart availability

## Step 9: Visualize Full Analysis

In [8]:
# Plot all available charts
fig_full = result_full.plot(
    template='processbehavior',
    width=1400,
    height=700,
    highlight_signals=True
)

fig_full.show()

## Step 10: Variance Decomposition (Residuals)

**Wheeler's Residuals** (R1-R5) break down variation into components:
- **R1**: Total deviation from grand mean
- **R2**: Time effects removed
- **R3**: Factor (lane/phase) effects removed
- **R4**: Both time and factor effects removed
- **R5**: Pure error (all systematic effects removed)

In [9]:
if result_full.has_residuals:
    print("Residuals Available: Yes\n")
    
    # Show first 10 observations with residuals
    display_cols = ['pull', 'lane', 'phase', 'fill_weight', 'R1', 'R2', 'R3', 'R4', 'R5']
    print("Sample Data with Residuals:")
    print(result_full.dataset[display_cols].head(10))
    
    # Calculate variance by residual type
    print("\n\nVariance Breakdown:")
    for col in ['R1', 'R2', 'R3', 'R4', 'R5']:
        var = result_full.residuals[col].var()
        print(f"  {col}: {var:.2f}")
else:
    print("Residuals not available for this SDS")

Residuals Available: Yes

Sample Data with Residuals:
    pull  lane  phase  fill_weight       R1   R2        R3       R4        R5
0      1     1      1       236.93 -0.85237  0.0 -0.656544 -0.52487  0.329044
8      2     1      1       239.67  1.88763  0.0  1.499706  0.05888  0.329044
16     3     1      1       239.67  1.88763  0.0  2.030956 -0.47237  0.329044
31     5     1      1       237.46 -0.32237  0.0 -0.915294  0.26388  0.329044
39     6     1      1       237.83  0.04763  0.0 -0.481544  0.20013  0.329044
47     7     1      1       238.66  0.87763  0.0  0.972206 -0.42362  0.329044
55     8     1      1       238.68  0.89763  0.0  0.550956  0.01763  0.329044
63     9     1      1       237.98  0.19763  0.0 -0.174044  0.04263  0.329044
71    10     1      1       238.03  0.24763  0.0 -0.072794 -0.00862  0.329044
79    11     1      1       237.71 -0.07237  0.0 -0.414044  0.01263  0.329044


Variance Breakdown:
  R1: 2.33
  R2: 0.00
  R3: 1.63
  R4: 0.09
  R5: 0.61


## Step 11: Main Effects Analysis

### Lane and Phase Effects
Do lanes differ systematically? Does phase matter?

In [10]:
if result_full.has_effects:
    if 'lane' in result_full.effects:
        print("Lane Main Effects:")
        print(result_full.effects['lane'])
        print()
    
    if 'phase' in result_full.effects:
        print("\nPhase Main Effects (Up vs Down):")
        print(result_full.effects['phase'])
        print("\nInterpretation:")
        print("  • Phase 1 (down) vs Phase 2 (up) comparison")
        print("  • Shows if arm position affects fill weight")

Lane Main Effects:
   lane  Main_Effect
0     1     0.615803
1     2     0.379297
2     3    -1.055162
3     4     0.058138


Phase Main Effects (Up vs Down):
   phase  Main_Effect
0      1    -0.384562
1      2     0.389467

Interpretation:
  • Phase 1 (down) vs Phase 2 (up) comparison
  • Shows if arm position affects fill weight


## Step 12: Interaction Effects

Do lanes behave differently in up vs down positions?

In [11]:
if result_full.has_interactions:
    print("Interaction Analysis Available: Yes\n")
    
    # Show average by lane-phase combination
    interaction_summary = result_full.dataset.groupby(['lane', 'phase'])['fill_weight'].agg(['mean', 'count'])
    print("Lane × Phase Combinations:")
    print(interaction_summary)
    print("\nInterpretation:")
    print("  • Each cell shows average fill weight for that lane/phase combo")
    print("  • Look for patterns: do some lanes change more between phases?")
else:
    print("Interactions not available for this configuration")

Interaction Analysis Available: Yes

Lane × Phase Combinations:
                  mean  count
lane phase                   
1    1      238.111414     99
     2      238.687857     98
2    1      237.394700    100
     2      238.944286     98
3    1      236.497980     99
     2      236.958776     98
4    1      237.587172     99
     2      238.096429     98

Interpretation:
  • Each cell shows average fill weight for that lane/phase combo
  • Look for patterns: do some lanes change more between phases?


## Step 13: Signal Detection (Full Analysis)

In [12]:
# Detect signals in full analysis
signals_full = result_full.detect_signals(
    chart='Xbar',
    rules='default',  # Only Rule 1 for Xbar (categorical comparisons)
    config=SignalConfig(min_observations=2)
)

print(f"Signal Detection - Full Analysis:")
print(f"  • Signals Found: {signals_full.has_signals}")
print(f"  • Total Count: {signals_full.count}")
print(f"  • Rules Applied: Rule 1 only (appropriate for Xbar charts)")

if signals_full.has_signals:
    print(f"\nSignal Details (first 15):")
    print(signals_full.violations[['rule_name', 'value', 'description']].head(15))

Signal Detection - Full Analysis:
  • Signals Found: True
  • Total Count: 5
  • Rules Applied: Rule 1 only (appropriate for Xbar charts)

Signal Details (first 15):
  rule_name    value                  description
0    rule_1  238.688  Point beyond control limits
1    rule_1  237.395  Point beyond control limits
2    rule_1  238.944  Point beyond control limits
3    rule_1  236.498  Point beyond control limits
4    rule_1  236.959  Point beyond control limits


## Step 14: Time Series Analysis with Stratified IMR

**Want to use Rules 2-4?** Use stratified IMR charts!

IMR charts show individual measurements over time - these ARE true time series where sequential rules apply.

In [13]:
# Create stratified IMR analysis (separate IMR chart for each lane-phase combination)
# Using the same study, just request IMR chart type
result_stratified = study_full.execute(chart='Imr')

print("Stratified IMR Analysis:")
print(f"  Charts created: {len(result_stratified.all_charts)}")
print(f"  Chart names: {result_stratified.all_charts[:8]}...")  # Show first 8

Stratified IMR Analysis:
  Charts created: 8
  Chart names: ['1_1', '1_2', '2_1', '2_2', '3_1', '3_2', '4_1', '4_2']...


### Detect Signals on IMR Charts (All Rules Apply)

In [14]:
# Detect signals on a specific IMR chart
# Get first IMR chart (all charts are IMR when stratified with chart_type='Imr')
imr_chart_name = result_stratified.all_charts[0]

print(f"\nAnalyzing: {imr_chart_name}")

signals_imr = result_stratified.detect_signals(
    chart=imr_chart_name,
    rules='standard',  # Rules 1-4 are valid for IMR (time series!)
    config=SignalConfig(min_observations=10)
)

print(f"  Signals Found: {signals_imr.has_signals}")
print(f"  Total Count: {signals_imr.count}")
print(f"  Rules Applied: 1-4 (appropriate for IMR time series)")

if signals_imr.has_signals:
    print(f"\nIMR Signal Details:")
    print(signals_imr.violations[['rule_name', 'value', 'description']].head(10))


Analyzing: 1_1
  Signals Found: True
  Total Count: 18
  Rules Applied: 1-4 (appropriate for IMR time series)

IMR Signal Details:
  rule_name   value                                   description
0    rule_1  232.31                   Point beyond control limits
1    rule_4  238.97  8+ consecutive points on same side of center
2    rule_4  238.65  8+ consecutive points on same side of center
3    rule_4  238.83  8+ consecutive points on same side of center
4    rule_4  238.75  8+ consecutive points on same side of center
5    rule_4  238.55  8+ consecutive points on same side of center
6    rule_1  234.38                   Point beyond control limits
7    rule_1  234.74                   Point beyond control limits
8    rule_7  238.42              15+ consecutive points in Zone C
9    rule_7  237.57              15+ consecutive points in Zone C


## Step 15: Export Comprehensive Results

Save everything to Excel for sharing with the team or further analysis.

In [15]:
# Export full analysis to Excel
result_full.to_excel(
    'fillweight_analysis_complete.xlsx',
    include_residuals=True,
    include_effects=True,
    include_interactions=True,
    include_full_dataset=True
)

print("✅ Results exported to: fillweight_analysis_complete.xlsx")
print("\nWorkbook includes:")
print("  • Summary sheet with analysis metadata")
print("  • Xbar chart data (subgroup means)")
print("  • Sbar chart data (subgroup variation)")
print("  • VAS Residuals (R1-R5 with original columns)")
print("  • Main Effects (lane and phase)")
print("  • Interactions (lane × phase)")
print("  • Full Dataset (all calculated values)")

Could not add Xbar chart image: 

Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome




Could not add Sbar chart image: 

Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome




✅ Results exported to: fillweight_analysis_complete.xlsx

Workbook includes:
  • Summary sheet with analysis metadata
  • Xbar chart data (subgroup means)
  • Sbar chart data (subgroup variation)
  • VAS Residuals (R1-R5 with original columns)
  • Main Effects (lane and phase)
  • Interactions (lane × phase)
  • Full Dataset (all calculated values)


---

# Summary: Iterative Analysis Workflow

## What We Learned

### Stage 1: Lane-Only Analysis (SDS 1)
✓ Quick overview of lane performance  
✓ 4 lanes with n≈200 observations each (full replication)
✓ Both Xbar AND S charts available
✓ Detect signals using Rule 1 (appropriate for Xbar/S)  
✓ Phase treated as replicates within each lane

### Stage 2: Lane + Phase Analysis (SDS 1)
✓ Understand effect of fill arm position  
✓ 8 subgroups (lane × phase) with n≈100 each (full replication)
✓ Both Xbar AND S charts available
✓ Full variance decomposition (R1-R5)  
✓ Separate main effects for lane and phase  
✓ Interaction analysis (lane × phase)  
✓ Comprehensive Excel export  

### Stage 3: Stratified IMR (Optional)
✓ True time series analysis per lane-phase combination  
✓ All WECO rules (1-4) applicable  
✓ Detect trends, runs, and patterns over time  

## Key Insights

**Chart Types Matter**:
- **Xbar/S**: Categorical comparisons → Only Rule 1 applies
- **IMR/R**: Time series → All rules (1-8) apply
- Understanding the data structure determines which rules make sense

**SDS Detection is Critical**:
- Subgroups = grouping variables (NOT grouping × time)
- Sample size n = observations per subgroup **across all time points**
- time_var is for sequencing/plotting, not subgroup definition
- Correct SDS detection → correct chart availability → valid conclusions

**Iterative Analysis is Powerful**:
1. Start simple (fewer grouping variables)
2. Identify areas needing investigation
3. Add complexity (more grouping variables)
4. Use stratification for time-series drill-down

**ProcessBehavior Adapts**:
- Automatically detects data structure (SDS)
- Recommends appropriate charts
- Applies correct WECO rules by default
- Handles missing values gracefully

## Next Steps

### Explore Further
1. **Custom Rule Sets**: Explicitly specify which rules to use
2. **Time Windows**: Analyze specific time ranges for process changes
3. **Multiple Responses**: Analyze other measurements (weight, volume, etc.)
4. **Compare Strategies**: Use different grouping strategies for different insights

### Production Use
- Use Wheeler's recommended minimum: 20+ observations for signal detection
- Document special causes when signals occur
- Recalculate limits after process changes
- Share Excel reports with the team

---

**Questions?** This workflow demonstrates how real practitioners use process behavior charts - understanding when different chart types and rules apply is key to correct analysis!